In [3]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow\\research'

In [4]:
%cd ..

c:\Users\Amal\Desktop\End to End project Mlops mlflow github\End-to-End-project-Mlops-Mlflow


C:\Users\Amal\AppData\Roaming\Python\Python39\site-packages\IPython\core\magics\osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [5]:
%pwd

'c:\\Users\\Amal\\Desktop\\End to End project Mlops mlflow github\\End-to-End-project-Mlops-Mlflow'

Test Normalisation and split data for Update the components

In [1]:
import pandas as pd

In [6]:
data = pd.read_csv("artifacts/data_ingestion/winequality-red.csv")

In [79]:
input_data = data.iloc[:,:11]

1. Normalisation not real time (temps différer) 
pour real time il faut par exple pour normalisation moyenne mean, max et min des valeurs de training uniquement (faire le split en 1 puis la normalisation) , stocker les valeurs mean, min et max et finalemnt normaliser les valeur de val et test en utilisant ces valeurs

In [ ]:
# get mean, max and min values of the input data
mean_input_data, max_input_data, min_input_data = input_data.mean(), input_data.max(), input_data.min()

In [ ]:
# normalise les donné d'entré avec la normalisation moyenne 
input_normalized_data = (input_data - mean_input_data)/(max_input_data-min_input_data)

In [82]:
input_normalized_data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,-0.081384,0.117931,-0.270976,-0.043754,-0.019143,-0.068661,-0.044056,0.077336,0.156604,-0.058772,-0.157382
1,-0.045986,0.241219,-0.270976,0.004191,0.017585,0.128522,0.072552,0.003915,-0.087491,0.013085,-0.095844
2,-0.045986,0.159027,-0.230976,-0.016357,0.007568,-0.012323,0.026616,0.018599,-0.040247,-0.004880,-0.095844
3,0.254899,-0.169740,0.289024,-0.043754,-0.020812,0.015846,0.047817,0.092021,-0.118987,-0.046796,-0.095844
4,-0.081384,0.117931,-0.270976,-0.043754,-0.019143,-0.068661,-0.044056,0.077336,0.156604,-0.058772,-0.157382
...,...,...,...,...,...,...,...,...,...,...,...
1594,-0.187579,0.049438,-0.190976,-0.036904,0.004229,0.227114,-0.008720,-0.135586,0.109360,-0.046796,0.011849
1595,-0.214127,0.015191,-0.170976,-0.023206,-0.042515,0.325705,0.016015,-0.119433,0.164478,0.060989,0.119541
1596,-0.178729,-0.012206,-0.140976,-0.016357,-0.019143,0.184860,-0.022854,-0.073912,0.085738,0.055001,0.088772
1597,-0.214127,0.080260,-0.150976,-0.036904,-0.020812,0.227114,-0.008720,-0.093736,0.203848,0.031049,-0.034305


In [99]:
# Concate target (output) to input normalised_data
data_normalized=pd.concat([input_normalized_data , data["quality"]], axis=1)
data_normalized

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,-0.081384,0.117931,-0.270976,-0.043754,-0.019143,-0.068661,-0.044056,0.077336,0.156604,-0.058772,-0.157382,5
1,-0.045986,0.241219,-0.270976,0.004191,0.017585,0.128522,0.072552,0.003915,-0.087491,0.013085,-0.095844,5
2,-0.045986,0.159027,-0.230976,-0.016357,0.007568,-0.012323,0.026616,0.018599,-0.040247,-0.004880,-0.095844,5
3,0.254899,-0.169740,0.289024,-0.043754,-0.020812,0.015846,0.047817,0.092021,-0.118987,-0.046796,-0.095844,6
4,-0.081384,0.117931,-0.270976,-0.043754,-0.019143,-0.068661,-0.044056,0.077336,0.156604,-0.058772,-0.157382,5
...,...,...,...,...,...,...,...,...,...,...,...,...
1594,-0.187579,0.049438,-0.190976,-0.036904,0.004229,0.227114,-0.008720,-0.135586,0.109360,-0.046796,0.011849,5
1595,-0.214127,0.015191,-0.170976,-0.023206,-0.042515,0.325705,0.016015,-0.119433,0.164478,0.060989,0.119541,6
1596,-0.178729,-0.012206,-0.140976,-0.016357,-0.019143,0.184860,-0.022854,-0.073912,0.085738,0.055001,0.088772,6
1597,-0.214127,0.080260,-0.150976,-0.036904,-0.020812,0.227114,-0.008720,-0.093736,0.203848,0.031049,-0.034305,5


2. Split data

In [54]:
from sklearn.model_selection import train_test_split

In [135]:
# Split the input normalized data into training and test sets : (0.75, 0.25)
train, test = train_test_split(data_normalized, test_size=0.25, random_state=42)

In [136]:
print(train.shape)
print(test.shape)

(1199, 12)
(400, 12)


Update the entity

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    all_schema: dict # read the file schema.yaml and save it in a dictionary format

Update the configuration manager

In [115]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories

In [116]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root]) # créer le dossier artifacts (correspond au artifacts_root dans fichier config.yaml)

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_path = config.data_path,
            all_schema = self.schema,
        )

        return data_transformation_config

Update the components

In [117]:
import os
from mlProject import logger
from sklearn.model_selection import train_test_split
import pandas as pd

In [137]:
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.data_normalized = None  # Initialiser comme None
        
    
    ## Note: You can add other data transformation techniques such as Scaler, PCA and all
    #You can perform all kinds of EDA in ML cycle here before passing this data to the model

    def Normalize_data (self):
        
        data = pd.read_csv(self.config.data_path)
        
        input_data = data.drop(self.config.all_schema.TARGET_COLUMN.name, axis=1)
        
         # get mean, max and min values of the input data
        mean_input_data, max_input_data, min_input_data = input_data.mean(), input_data.max(), input_data.min()

        # normalize les donné d'entré avec la normalisation moyenne 
        input_normalized_data = (input_data - mean_input_data)/(max_input_data-min_input_data)

        # Concate target (output) to input normalised_data
        self.data_normalized = pd.concat([input_normalized_data , data[self.config.all_schema.TARGET_COLUMN.name]], axis=1)

        return self.data_normalized
        
        
    def train_test_splitting(self, data_normalized):
  
       # Split the input normalized data into training and test sets : (0.75, 0.25)
        train, test= train_test_split(data_normalized, test_size=0.25, random_state=42)
        
        train.to_csv(os.path.join(self.config.root_dir, "train.csv"),index = False)
        test.to_csv(os.path.join(self.config.root_dir, "test.csv"),index = False)

        logger.info("Split data into training and test sets")
        logger.info("train shape: " + str(train.shape))  
        logger.info("test shape: "+ str(test.shape))

Update the pipline

In [138]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_normalized = data_transformation. Normalize_data()
    data_transformation.train_test_splitting(data_normalized)
except Exception as e:
    raise e

[2026-06-02 16:16:30,631: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-06-02 16:16:30,658: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-02 16:16:30,682: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-06-02 16:16:30,702: INFO: common: created directory at: artifacts]
[2026-06-02 16:16:30,723: INFO: common: created directory at: artifacts/data_transformation]
[2026-06-02 16:16:31,602: INFO: 1546428402: Split data into training and test sets]
[2026-06-02 16:16:31,604: INFO: 1546428402: train shape: (1199, 12)]
[2026-06-02 16:16:31,608: INFO: 1546428402: test shape: (400, 12)]
